# Evaluate Gemini 2.5 Flash using MedEvidence Data 

Only evaluate the questions that do not need full text


In [8]:
from utils import render_prompt
import pandas as pd
from models.gemini import Gemini
from utils import load_json_file, save_dataset_to_json, load_jsonl_file
import seaborn as sns
import matplotlib.pyplot as plt
from Bio import Entrez
import ast
from tqdm import tqdm


Entrez.email = 'yun.hy@northeastern.edu'
EVAL_MODEL_TEMPERATURE = 0.0
EVIDENCE_DIRECTION_PROMPT_TEMPLATE_NAMES = "evidence_direction_question"

## Submit Job to Gemini

In [2]:
# load medevidence dataset
medevidence_file_path = "../data/medEvidence.csv"
medevidence_df = pd.read_csv(medevidence_file_path)

medevidence_df.shape

(284, 11)

In [3]:
# filter out rows where fulltext_required is yes. Only keep rows where fulltext_required is no
filtered_medevidence_df = medevidence_df[medevidence_df['fulltext_required'] == 'no']
filtered_medevidence_df.shape

(216, 11)

In [4]:
def get_abstract(pmid):
    try:
        # Fetch the record from the 'pubmed' database in XML format
        handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
        records = Entrez.read(handle)
        handle.close()

        # Drill down into the XML structure to find the abstract
        # Note: Some articles may not have an abstract
        article = records['PubmedArticle'][0]['MedlineCitation']['Article']
        
        if 'Abstract' in article:
            # Abstracts can be split into multiple sections (Background, Methods, etc.)
            abstract_list = article['Abstract']['AbstractText']
            return " ".join(abstract_list)
        else:
            return "Abstract not available for this PMID."

    except Exception as e:
        return f"Error: {str(e)}"

In [5]:
# format data for model evaluation
data = filtered_medevidence_df.to_dict(orient='records')
formatted_input_for_model_evaluator = {}
for row in tqdm(data):
    question = row['question']
    sources = ast.literal_eval(row['sources'])
    question_id = row['question_id']

    new_sources = []
    for pmid, source in sources.items():
        abstract = get_abstract(pmid) # this is needed since not all the content in the dataset is just abstracts. some are full text which is not what we want.
        if abstract.startswith("Error:") or abstract == "Abstract not available for this PMID.":
            print(f"Warning: Could not retrieve abstract for PMID {pmid}. Skipping this source.")
            continue
        new_sources.append({"article_id": source, "summary": abstract})

    eval_direction_input = render_prompt(EVIDENCE_DIRECTION_PROMPT_TEMPLATE_NAMES, template_dir="./prompts", question=question, context=new_sources)
    formatted_input_for_model_evaluator[f"{question_id}"] = eval_direction_input

100%|██████████| 216/216 [03:19<00:00,  1.09it/s]


In [6]:
# save formatted input for model evaluator to jsonl file
formatted_input_list = [{"question_id": question_id, "input": input} for question_id, input in formatted_input_for_model_evaluator.items()]
save_dataset_to_json(formatted_input_list, "../data/medevidence_formatted_eval_input.jsonl", jsonl=True)

In [9]:
formatted_input_list = load_jsonl_file("../data/medevidence_formatted_eval_input.jsonl")
formatted_input_for_model_evaluator = {item['question_id']: item['input'] for item in formatted_input_list}

In [10]:
# get the total number of words in the formatted input for model evaluator
# approximation for the total number of tokens we will be sending to Gemini.
# this is helpful in figuring out the rate limit for the batch api
total_words = sum(len(input.split()) for input in formatted_input_for_model_evaluator.values())
print(f"Total number of words in formatted input for model evaluator: {total_words}")


Total number of words in formatted input for model evaluator: 799179


In [11]:
# Since 100 tokens is equal to about 60-80 English words. (citation: https://ai.google.dev/gemini-api/docs/tokens)
# we can estimate the total number of tokens by dividing the total number of words by 70 (the average of 60 and 80)
estimated_total_tokens = total_words / 70
print(f"Estimated total number of tokens in formatted input for model evaluator: {estimated_total_tokens}")

Estimated total number of tokens in formatted input for model evaluator: 11416.842857142858


In [ ]:
# load the evaluation model
eval_model = Gemini("2.5")

eval_model.submit_batch(formatted_input_for_model_evaluator, EVAL_MODEL_TEMPERATURE)

## Get results

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=gemini_api_key)

In [108]:
BATCH_ID = "batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y"

In [123]:
print(f"Retrieving status for job: {BATCH_ID}")

batch_job = gemini_client.batches.get(name=BATCH_ID)
print(f"Current state: {batch_job.state.name}")
print(f"Job details: {batch_job}")

Retrieving status for job: batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y
Current state: JOB_STATE_RUNNING
Job details: name='batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y' display_name=None state=<JobState.JOB_STATE_RUNNING: 'JOB_STATE_RUNNING'> error=None create_time=datetime.datetime(2026, 2, 24, 18, 47, 38, 274769, tzinfo=TzInfo(UTC)) start_time=None end_time=None update_time=datetime.datetime(2026, 2, 24, 18, 52, 8, 224752, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=None completion_stats=None


In [105]:
def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch_job = gemini_client.batches.get(name=batch_name)

    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    results = {}
    for line in file_content.splitlines():
        if line:
            record = json.loads(line)
            custom_id = record["key"]
            response = record["response"]["candidates"][0]
            output_text = response["content"]["parts"][0]["text"]
            results[custom_id] = output_text
    return results

In [23]:
results = get_batch_results(BATCH_ID)

save_dataset_to_json(results, "../data/medevidence_evaluation_results.json")

Results are in file: files/batch-jy8bq5pu3sdc4t3oquabytams3jeumjjqt6v



## Analyze Result

In [24]:
results = load_json_file("../data/medevidence_evaluation_results.json")

print(len(results))

640


In [ ]:
medevidence_file_path = "../data/medEvidence.csv"
medevidence_df = pd.read_csv(medevidence_file_path)
filtered_medevidence_df = medevidence_df[medevidence_df['fulltext_required'] == 'no']
data = filtered_medevidence_df.to_dict(orient='records')

In [ ]:
def extract_answer(text: str) -> str:
    """
    Extracts only nswer if answer exists else return the full text

    :param text: string of text

    :return: str
    """
    delimiter = "**Answer**:"
    parts = text.split(delimiter, 1)

    # If len is > 1, delimiter existed; return second part, stripped of whitespace
    if len(parts) > 1:
        return parts[1].strip()

    return text.lower()

In [ ]:
for row in data:
    question_id = row['question_id']
    if question_id in results:
        response = results[f"req-{question_id}"]
        row["gemini_output"] = extract_answer(response)

In [ ]:
# % of valid gemini output
# the output must be one of the following: higher, lower, no difference, insufficient data, uncertain effect
valid_outputs = {"higher", "lower", "no difference", "insufficient data", "uncertain effect"}
valid_count = 0
for row in data:
    if "gemini_output" in row and row["gemini_output"] in valid_outputs:
        valid_count += 1

print(f"Valid outputs: {valid_count}/{len(data)} ({valid_count/len(data)*100:.2f}%)")

In [ ]:
# % of accuracy of gemini output compared to the human answer which is the "answer" key in the dataset
accurate_count = 0
for row in data:
    if "gemini_output" in row and row["gemini_output"] == row["answer"]:
        accurate_count += 1
        
print(f"Accurate outputs: {accurate_count}/{len(data)} ({accurate_count/len(data)*100:.2f}%)")

In [ ]:
# get the recall by each answer category
answer_categories = valid_outputs
matrix_by_category = {category: {"tp": 0, "fn": 0} for category in answer_categories}
for row in data:
    if "gemini_output" in row and row["answer"] in answer_categories:
        human_answer = row["answer"]
        if row["gemini_output"] == human_answer:
            matrix_by_category[human_answer]["tp"] += 1
        else:
            matrix_by_category[human_answer]["fn"] += 1

recall_data = []
for category, counts in matrix_by_category.items():
    tp = counts["tp"]
    fn = counts["fn"]
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_data.append({"category": category, "recall": recall})
    print(f"Category: {category}, Recall: {recall:.2f} ({tp}/{tp + fn})")

In [ ]:
output_counts = {category: 0 for category in valid_outputs}
for row in data:
    if "gemini_output" in row and row["gemini_output"] in valid_outputs:
        output_counts[row["gemini_output"]] += 1

plt.figure(figsize=(10, 6))
sns.barplot(x=list(output_counts.keys()), y=list(output_counts.values()))
plt.title("Distribution of Gemini Outputs")
plt.xlabel("Gemini Output Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
recall_df = pd.DataFrame(recall_data)
plt.figure(figsize=(10, 6))
sns.barplot(x="category", y="recall", data=recall_df)
plt.title("Recall by Answer Category")
plt.xlabel("Answer Category")
plt.ylabel("Recall")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# seaborn bar plot of accuracy over time based on review_year

accuracy_by_year = {}
for row in data:
    if "gemini_output" in row and row["answer"] in valid_outputs:
        review_year = row["review_year"]
        if review_year not in accuracy_by_year:
            accuracy_by_year[review_year] = {"tp": 0, "total": 0}
        accuracy_by_year[review_year]["total"] += 1
        if row["gemini_output"] == row["answer"]:
            accuracy_by_year[review_year]["tp"] += 1

results_df = pd.DataFrame([
    {"review_year": year, "accuracy": counts["tp"] / counts["total"] if counts["total"] > 0 else 0.0, "model": "gemini-2.5"}
     for year, counts in accuracy_by_year.items()
])

sns.barplot(x='review_year', y='accuracy', hue='model', data=results_df)
plt.title('Accuracy Over Time by Model')
plt.xlabel('Review Year')
plt.ylabel('Accuracy')
plt.show()